# Gaussian GRAPE Data Analysis

Load, validate, summarize, plot, and save first-level analysis outputs for the Gaussian GRAPE crosstalk sweep.


In [ ]:
import json
import math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


RESULTS_ROOT = Path("PiecewiseGRAPEResults_selected_v1")
if not RESULTS_ROOT.exists() and Path("Scripts/PiecewiseGRAPEResults_selected_v1").exists():
    RESULTS_ROOT = Path("Scripts/PiecewiseGRAPEResults_selected_v1")

ANALYSIS_OUT = Path("PiecewiseGRAPEAnalysis_stage2_selected_v1")
if RESULTS_ROOT.parts and RESULTS_ROOT.parts[0] == "Scripts":
    ANALYSIS_OUT = Path("Scripts/PiecewiseGRAPEAnalysis_stage2_selected_v1")

# None means discover the actual eta_* folders. This supports sanity subsets like eta_000, eta_006, ...
N_ETA_EXPECTED = None
SUCCESS_THRESHOLD_ANALYSIS = 0.999

# Match the project notebook's 0.1-GHz unit convention for eta/(2*pi*GHz) labels.
GHz = globals().get("GHz", 1e8)
ns = globals().get("ns", 1e-9)

figure_width = 10
figure_height = 6


# Load and Validate Sweep Data

These helpers validate the expected eta-folder structure, load root-level summaries, and load restart-level records.


In [ ]:
def safe_float(x, default=np.nan):
    try:
        if x is None:
            return default
        value = float(x)
        return value if math.isfinite(value) else default
    except Exception:
        return default


def safe_int(x, default=0):
    try:
        if x is None:
            return default
        return int(x)
    except Exception:
        return default


def safe_bool(x, default=False):
    if isinstance(x, bool):
        return x
    if x is None:
        return default
    if isinstance(x, str):
        value = x.strip().lower()
        if value in {"true", "1", "yes", "y"}:
            return True
        if value in {"false", "0", "no", "n"}:
            return False
        return default
    try:
        return bool(x)
    except Exception:
        return default


def sanitize_stop_reason(reason):
    if reason is None:
        return "missing"
    reason = str(reason).strip()
    if reason.startswith("exception:"):
        return "exception"
    if not reason:
        return "missing"
    safe = []
    for char in reason.lower():
        if char.isalnum() or char == "_":
            safe.append(char)
        elif char in {" ", "-", "/", ".", ":"}:
            safe.append("_")
    sanitized = "".join(safe).strip("_")
    while "__" in sanitized:
        sanitized = sanitized.replace("__", "_")
    return sanitized or "unknown"


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(make_json_safe(key)): make_json_safe(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [make_json_safe(value) for value in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.ndarray):
        return make_json_safe(obj.tolist())
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        value = float(obj)
        return value if math.isfinite(value) else None
    return obj


def _json_load(path, warnings=None):
    try:
        with Path(path).open("r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        if warnings is not None:
            warnings.append(f"Could not load JSON {path}: {exc}")
        return None




In [ ]:
def eta_index_from_dir(eta_dir):
    try:
        return int(Path(eta_dir).name.split("_")[-1])
    except Exception:
        return 0


def discover_eta_dirs(results_root, n_eta_expected=None):
    results_root = Path(results_root)
    if not results_root.exists():
        return []
    discovered = sorted(
        [path for path in results_root.glob("eta_*") if path.is_dir()],
        key=eta_index_from_dir,
    )
    if n_eta_expected is None:
        return discovered

    expected = [results_root / f"eta_{eta_index:03d}" for eta_index in range(int(n_eta_expected))]
    extra = [path for path in discovered if path not in expected]
    return sorted(expected + extra, key=eta_index_from_dir)


def validate_results_structure(results_root, n_eta_expected=None):
    results_root = Path(results_root)
    required_eta_files = [
        "results.json",
        "summary.json",
        "summary_row.json",
        "metadata.json",
    ]
    validation = {
        "results_root": str(results_root),
        "results_root_exists": results_root.exists(),
        "n_eta_expected": None if n_eta_expected is None else int(n_eta_expected),
        "eta_folders_found": [],
        "eta_folders_missing": [],
        "missing_files": [],
        "present_files": [],
        "warnings": [],
        "notes": [],
        "optional_missing_files": [],
        "root_files": {},
    }

    if not results_root.exists():
        validation["warnings"].append(f"Results root does not exist: {results_root}")
        return validation

    eta_dirs = discover_eta_dirs(results_root, n_eta_expected=n_eta_expected)
    if not eta_dirs:
        validation["warnings"].append(f"No eta_* folders found in {results_root}")

    for eta_dir in eta_dirs:
        if eta_dir.exists():
            validation["eta_folders_found"].append(eta_dir.name)
        else:
            validation["eta_folders_missing"].append(eta_dir.name)
            validation["warnings"].append(f"Missing eta folder: {eta_dir}")
            continue

        for filename in required_eta_files:
            path = eta_dir / filename
            if path.exists():
                validation["present_files"].append(str(path))
            else:
                validation["missing_files"].append(str(path))
                validation["warnings"].append(f"Missing file: {path}")

    # Root-level grid summaries are optional.
    for filename in ["grid_summary.json", "grid_summary.csv"]:
        path = results_root / filename
        validation["root_files"][filename] = path.exists()
        if not path.exists():
            validation["optional_missing_files"].append(str(path))

    if validation["optional_missing_files"]:
        validation["notes"].append(
            "Root-level grid_summary.json/csv are missing; Stage 2 will reconstruct "
            "grid-level summaries from restart records."
        )

    found_indices = [eta_index_from_dir(name) for name in validation["eta_folders_found"]]
    validation["eta_indices_found"] = found_indices
    validation["n_eta_found"] = len(validation["eta_folders_found"])
    validation["n_missing_files"] = len(validation["missing_files"])
    validation["ok"] = (
        validation["results_root_exists"]
        and not validation["eta_folders_missing"]
        and not validation["missing_files"]
    )
    return validation


def load_grid_summary(results_root):
    results_root = Path(results_root)
    json_path = results_root / "grid_summary.json"
    csv_path = results_root / "grid_summary.csv"

    grid_summary_json_df = pd.DataFrame()
    grid_summary_csv_df = pd.DataFrame()

    if json_path.exists():
        data = _json_load(json_path, warnings=[])
        if data is not None:
            grid_summary_json_df = pd.DataFrame(data)

    if csv_path.exists():
        try:
            grid_summary_csv_df = pd.read_csv(csv_path)
        except Exception:
            grid_summary_csv_df = pd.DataFrame()

    return grid_summary_json_df, grid_summary_csv_df


def load_restart_records(results_root, n_eta_expected=None):
    results_root = Path(results_root)
    records = []
    warnings = []

    for eta_dir in discover_eta_dirs(results_root, n_eta_expected=n_eta_expected):
        eta_index = eta_index_from_dir(eta_dir)
        results_path = eta_dir / "results.json"
        if not results_path.exists():
            warnings.append(f"Missing restart results: {results_path}")
            continue

        eta_records = _json_load(results_path, warnings=warnings)
        if eta_records is None:
            continue
        if not isinstance(eta_records, list):
            warnings.append(f"Expected list in {results_path}, got {type(eta_records).__name__}")
            continue

        for record in eta_records:
            if not isinstance(record, dict):
                warnings.append(f"Skipping non-dict record in {results_path}")
                continue
            record = dict(record)
            record["eta_index"] = safe_int(record.get("eta_index"), default=eta_index)
            record["eta_dir"] = str(eta_dir)
            record["source_file"] = str(results_path)
            records.append(record)

    if warnings:
        print("Restart-load warnings:")
        for warning in warnings:
            print(" -", warning)
    return records


Convert raw restart dictionaries into a tidy DataFrame, extract optimized Gaussian parameters, and compute per-eta summary statistics.


In [ ]:
def extract_x_opt_scaled(record):
    x_opt_scaled = record.get("x_opt_scaled")
    if isinstance(x_opt_scaled, (list, tuple)) and len(x_opt_scaled) > 0:
        return [safe_float(value) for value in x_opt_scaled]

    c_opt = record.get("c_opt")
    try:
        c_array = np.asarray(c_opt, dtype=float)
        if c_array.shape == (4, 2):
            return [
                safe_float(value) / (ns if column == 1 else 1.0)
                for row in c_array
                for column, value in enumerate(row)
            ]
        if c_array.ndim == 3 and c_array.shape[0] == 4 and c_array.shape[2] >= 3:
            return [
                safe_float(value) / (ns if param_index in {1, 2} else 1.0)
                for channel in range(c_array.shape[0])
                for pulse_index in range(c_array.shape[1])
                for param_index, value in enumerate(c_array[channel, pulse_index, :3])
            ]
        if c_array.ndim == 2 and c_array.shape[0] == 4 and c_array.shape[1] > 2:
            return [safe_float(value) for value in c_array.reshape(-1)]
    except Exception:
        pass

    n_params = safe_int(record.get("n_optimized_parameters"), default=0)
    return [np.nan] * n_params if n_params > 0 else []


def parameter_names_for_record(record, x_values):
    n_params = len(x_values)
    pulse_family = str(record.get("pulse_family", ""))
    n_channels = safe_int(record.get("n_channels"), default=4)
    n_gaussians = safe_int(record.get("n_gaussians_per_channel"), default=0)
    params_per_gaussian = safe_int(record.get("params_per_gaussian"), default=0)

    if n_params == 8 and (not n_gaussians or params_per_gaussian in {0, 2}):
        return [
            "A0", "width0_ns",
            "A1", "width1_ns",
            "A2", "width2_ns",
            "A3", "width3_ns",
        ]

    if (
        "two_gaussian" in pulse_family
        or params_per_gaussian == 3
        or (n_channels == 4 and n_params % 12 == 0)
    ):
        if not n_gaussians:
            n_gaussians = max(1, n_params // (max(n_channels, 1) * 3))
        names = []
        for channel in range(n_channels):
            for pulse_index in range(n_gaussians):
                names.extend([
                    f"C{channel}_g{pulse_index}_A",
                    f"C{channel}_g{pulse_index}_width_ns",
                    f"C{channel}_g{pulse_index}_center_ns",
                ])
        return names[:n_params]

    if "piecewise" in pulse_family or (n_channels > 0 and n_params > 8 and n_params % n_channels == 0):
        n_time_bins = safe_int(record.get("n_time_bins"), default=0)
        if not n_time_bins:
            l_value = safe_int(record.get("L"), default=0)
            if l_value and n_channels * l_value == n_params:
                n_time_bins = l_value
            else:
                n_time_bins = max(1, n_params // max(n_channels, 1))
        names = []
        for channel in range(n_channels):
            for time_bin in range(n_time_bins):
                names.append(f"C{channel}_bin{time_bin:02d}_A")
        return names[:n_params]

    return [f"param_{idx:02d}" for idx in range(n_params)]


def parameter_columns_from_dataframe(df):
    metadata_columns = {
        "eta_index", "eta", "eta_over_2pi_GHz", "restart", "seed", "seed_type", "seed_subindex",
        "used_physical_seed", "initial_fidelity", "final_fidelity", "valid_final_fidelity",
        "success_recorded", "success_analysis", "trap_analysis", "stop_reason", "stop_reason_sanitized",
        "count", "ended_at_bounds", "invalid_gradient_evals", "trace_len", "trace_initial",
        "trace_final", "trace_best", "trace_improvement", "trace_monotone_nondec", "pulse_family",
        "optimizer", "objective", "target", "T", "T_ns", "L", "max_iter", "eps0", "dx0",
        "improvement_threshold", "success_threshold", "optimizer_success_threshold", "n_optimized_parameters",
        "n_channels", "n_gaussians_per_channel", "params_per_gaussian", "n_time_bins", "eta_dir", "source_file",
    }
    return [col for col in df.columns if col not in metadata_columns and pd.api.types.is_numeric_dtype(df[col])]


def extract_trace_features(record):
    trace = record.get("record_cost") or []
    values = [safe_float(value) for value in trace]
    finite_values = [value for value in values if math.isfinite(value)]

    if not finite_values:
        return {
            "trace_len": 0,
            "trace_initial": np.nan,
            "trace_final": np.nan,
            "trace_best": np.nan,
            "trace_improvement": np.nan,
            "trace_monotone_nondec": False,
        }

    monotone = True
    previous = finite_values[0]
    for value in finite_values[1:]:
        if value + 1e-12 < previous:
            monotone = False
            break
        previous = value

    return {
        "trace_len": len(finite_values),
        "trace_initial": finite_values[0],
        "trace_final": finite_values[-1],
        "trace_best": max(finite_values),
        "trace_improvement": finite_values[-1] - finite_values[0],
        "trace_monotone_nondec": monotone,
    }


def _eta_over_2pi_ghz(eta):
    eta = safe_float(eta)
    denom = 2 * np.pi * GHz
    if not math.isfinite(eta) or denom == 0:
        return np.nan
    return eta / denom


def restart_records_to_dataframe(records, success_threshold=0.99):
    rows = []
    for record in records:
        x = extract_x_opt_scaled(record)
        parameter_names = parameter_names_for_record(record, x)
        trace_features = extract_trace_features(record)
        final_fidelity = safe_float(record.get("final_fidelity"))
        valid_final = math.isfinite(final_fidelity)
        success_analysis = bool(valid_final and final_fidelity >= success_threshold)
        stop_reason = record.get("stop_reason")

        row = {
            "eta_index": safe_int(record.get("eta_index")),
            "eta": safe_float(record.get("eta")),
            "eta_over_2pi_GHz": _eta_over_2pi_ghz(record.get("eta")),
            "restart": safe_int(record.get("restart")),
            "seed": safe_int(record.get("seed")),
            "seed_type": str(record.get("seed_type", "")),
            "seed_subindex": safe_int(record.get("seed_subindex")),
            "used_physical_seed": safe_bool(record.get("used_physical_seed")),
            "initial_fidelity": safe_float(record.get("initial_fidelity")),
            "final_fidelity": final_fidelity,
            "valid_final_fidelity": valid_final,
            "success_recorded": safe_bool(record.get("success")),
            "success_analysis": success_analysis,
            "trap_analysis": bool(not success_analysis),
            "stop_reason": str(stop_reason) if stop_reason is not None else "",
            "stop_reason_sanitized": sanitize_stop_reason(stop_reason),
            "count": safe_int(record.get("count")),
            "ended_at_bounds": safe_bool(record.get("ended_at_bounds")),
            "invalid_gradient_evals": safe_int(record.get("invalid_gradient_evals")),
            "pulse_family": str(record.get("pulse_family", "")),
            "optimizer": str(record.get("optimizer", "")),
            "objective": str(record.get("objective", "")),
            "target": str(record.get("target", "")),
            "T": safe_float(record.get("T")),
            "T_ns": safe_float(record.get("T_ns")),
            "L": safe_int(record.get("L")),
            "max_iter": safe_int(record.get("max_iter")),
            "eps0": safe_float(record.get("eps0")),
            "dx0": safe_float(record.get("dx0")),
            "improvement_threshold": safe_float(record.get("improvement_threshold")),
            "success_threshold": safe_float(record.get("success_threshold")),
            "optimizer_success_threshold": safe_float(record.get("optimizer_success_threshold")),
            "n_optimized_parameters": safe_int(record.get("n_optimized_parameters"), default=len(x)),
            "n_channels": safe_int(record.get("n_channels"), default=4),
            "n_gaussians_per_channel": safe_int(record.get("n_gaussians_per_channel")),
            "params_per_gaussian": safe_int(record.get("params_per_gaussian")),
            "n_time_bins": safe_int(record.get("n_time_bins"), default=(safe_int(record.get("L"), default=0) if "piecewise" in str(record.get("pulse_family", "")) else 0)),
            "eta_dir": record.get("eta_dir", ""),
            "source_file": record.get("source_file", ""),
        }
        for name, value in zip(parameter_names, x):
            row[name] = value
        row.update(trace_features)
        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values(["eta_index", "restart"]).reset_index(drop=True)


In [ ]:
def _series_quantile(series, q):
    finite = pd.to_numeric(series, errors="coerce").dropna()
    return float(finite.quantile(q)) if len(finite) else np.nan


def summarize_by_eta_from_records(df, success_threshold=0.99):
    if df.empty:
        return pd.DataFrame()

    stop_reasons = sorted(df["stop_reason_sanitized"].dropna().unique())
    parameter_columns = parameter_columns_from_dataframe(df)
    rows = []

    for eta_index, group in df.groupby("eta_index", sort=True):
        valid_group = group[group["valid_final_fidelity"]]
        fidelities = pd.to_numeric(valid_group["final_fidelity"], errors="coerce").dropna()
        success_mask = group["success_analysis"].fillna(False)
        trap_mask = group["trap_analysis"].fillna(False)
        physical = group[group["used_physical_seed"]]
        random = group[~group["used_physical_seed"]]

        best_row = None
        if len(valid_group):
            best_row = valid_group.loc[valid_group["final_fidelity"].idxmax()]

        row = {
            "eta_index": int(eta_index),
            "eta": safe_float(group["eta"].iloc[0]),
            "eta_over_2pi_GHz": safe_float(group["eta_over_2pi_GHz"].iloc[0]),
            "n_restarts": int(len(group)),
            "n_valid": int(len(valid_group)),
            "best_fidelity": float(fidelities.max()) if len(fidelities) else np.nan,
            "mean_fidelity": float(fidelities.mean()) if len(fidelities) else np.nan,
            "median_fidelity": float(fidelities.median()) if len(fidelities) else np.nan,
            "std_fidelity": float(fidelities.std(ddof=1)) if len(fidelities) > 1 else np.nan,
            "min_fidelity": float(fidelities.min()) if len(fidelities) else np.nan,
            "q25_fidelity": _series_quantile(fidelities, 0.25),
            "q75_fidelity": _series_quantile(fidelities, 0.75),
            "success_count": int(success_mask.sum()),
            "success_rate": float(success_mask.mean()) if len(group) else np.nan,
            "trap_count": int(trap_mask.sum()),
            "trap_ratio": float(trap_mask.mean()) if len(group) else np.nan,
            "physical_seed_count": int(len(physical)),
            "physical_seed_success_count": int(physical["success_analysis"].sum()) if len(physical) else 0,
            "physical_seed_success_rate": float(physical["success_analysis"].mean()) if len(physical) else np.nan,
            "random_seed_count": int(len(random)),
            "random_seed_success_count": int(random["success_analysis"].sum()) if len(random) else 0,
            "random_seed_success_rate": float(random["success_analysis"].mean()) if len(random) else np.nan,
            "failed_gradient_runs": int((group["stop_reason_sanitized"] == "invalid_gradient").sum()),
            "exception_runs": int((group["stop_reason_sanitized"] == "exception").sum()),
            "runs_ending_at_bounds": int(group["ended_at_bounds"].sum()),
            "bound_fraction": float(group["ended_at_bounds"].mean()) if len(group) else np.nan,
            "mean_iterations": float(pd.to_numeric(group["count"], errors="coerce").mean()),
            "median_iterations": float(pd.to_numeric(group["count"], errors="coerce").median()),
            "max_iterations": int(pd.to_numeric(group["count"], errors="coerce").max()),
            "mean_invalid_gradient_evals": float(pd.to_numeric(group["invalid_gradient_evals"], errors="coerce").mean()),
            "pulse_family": str(group["pulse_family"].dropna().iloc[0]) if "pulse_family" in group and len(group["pulse_family"].dropna()) else "",
            "n_optimized_parameters": safe_int(group["n_optimized_parameters"].dropna().iloc[0]) if "n_optimized_parameters" in group and len(group["n_optimized_parameters"].dropna()) else len(parameter_columns),
        }

        for reason in stop_reasons:
            row[f"stop_{reason}_count"] = int((group["stop_reason_sanitized"] == reason).sum())

        if best_row is not None:
            row.update({
                "best_restart": safe_int(best_row["restart"]),
                "best_seed": safe_int(best_row["seed"]),
                "best_seed_type": str(best_row.get("seed_type", "")),
                "best_used_physical_seed": safe_bool(best_row["used_physical_seed"]),
                "best_stop_reason": best_row["stop_reason"],
                "best_count": safe_int(best_row["count"]),
            })
            for column in parameter_columns:
                row[f"best_{column}"] = safe_float(best_row.get(column))
        else:
            row.update({
                "best_restart": np.nan,
                "best_seed": np.nan,
                "best_seed_type": "",
                "best_used_physical_seed": False,
                "best_stop_reason": "",
                "best_count": np.nan,
            })
            for column in parameter_columns:
                row[f"best_{column}"] = np.nan
        rows.append(row)

    return pd.DataFrame(rows).sort_values("eta_index").reset_index(drop=True)


def load_summary_rows(results_root, n_eta_expected=None):
    results_root = Path(results_root)
    rows = []
    warnings = []
    for eta_dir in discover_eta_dirs(results_root, n_eta_expected=n_eta_expected):
        path = eta_dir / "summary_row.json"
        if not path.exists():
            warnings.append(f"Missing summary_row.json: {path}")
            continue
        row = _json_load(path, warnings=warnings)
        if isinstance(row, dict):
            rows.append(row)
    if warnings:
        print("Summary-row warnings:")
        for warning in warnings:
            print(" -", warning)
    return pd.DataFrame(rows)


def compare_computed_vs_saved_summary(computed_df, saved_df):
    compare_columns = [
        "n_restarts",
        "best_fidelity",
        "mean_fidelity",
        "median_fidelity",
        "success_count",
        "success_rate",
        "failed_gradient_runs",
        "runs_ending_at_bounds",
        "mean_iterations",
        "median_iterations",
    ]
    report = {
        "available": not computed_df.empty and not saved_df.empty,
        "columns_compared": [],
        "max_abs_differences": {},
        "warnings": [],
    }

    if computed_df.empty:
        report["warnings"].append("Computed summary DataFrame is empty.")
        return report
    if saved_df.empty:
        report["warnings"].append("Saved summary rows DataFrame is empty.")
        return report

    if "eta_index" not in saved_df.columns:
        report["warnings"].append("Saved summary rows do not contain eta_index.")
        return report

    merged = computed_df.merge(saved_df, on="eta_index", suffixes=("_computed", "_saved"))
    if merged.empty:
        report["warnings"].append("No overlapping eta_index values between computed and saved summaries.")
        return report

    for column in compare_columns:
        computed_col = f"{column}_computed"
        saved_col = f"{column}_saved"
        if computed_col not in merged.columns or saved_col not in merged.columns:
            continue
        computed_values = pd.to_numeric(merged[computed_col], errors="coerce")
        saved_values = pd.to_numeric(merged[saved_col], errors="coerce")
        diff = (computed_values - saved_values).abs()
        max_diff = float(diff.max()) if len(diff.dropna()) else np.nan
        report["columns_compared"].append(column)
        report["max_abs_differences"][column] = max_diff
        if math.isfinite(max_diff) and max_diff > 1e-9:
            report["warnings"].append(f"{column} differs by max abs {max_diff:.3e}")

    return report


# Plots

Generate the first-level Stage 2 plots: fidelity trends, success/trap rates, restart scatter, optimizer effort, stop reasons, best Gaussian parameters, and representative convergence traces.


In [ ]:
def _maybe_show():
    try:
        from IPython import get_ipython

        if get_ipython() is not None:
            plt.show()
        else:
            plt.close()
    except Exception:
        plt.close()


def _save_current_figure(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    _maybe_show()


def plot_fidelity_vs_eta(summary_df, figures_dir):
    if summary_df.empty:
        return None
    plt.figure(figsize=(8, 5))
    x = summary_df["eta_over_2pi_GHz"]
    for column, label in [
        ("best_fidelity", "best"),
        ("mean_fidelity", "mean"),
        ("median_fidelity", "median"),
    ]:
        if column in summary_df.columns:
            plt.plot(x, summary_df[column], marker="o", label=label)
    plt.axhline(SUCCESS_THRESHOLD_ANALYSIS, color="red", linestyle="--", label=f"threshold {SUCCESS_THRESHOLD_ANALYSIS}")
    plt.xlabel("eta / (2 pi GHz)")
    plt.ylabel("Fidelity")
    plt.title("Piecewise GRAPE fidelity vs crosstalk")
    plt.grid(True, alpha=0.3)
    plt.legend()
    path = Path(figures_dir) / "fidelity_vs_eta.png"
    _save_current_figure(path)
    return path


def plot_success_trap_vs_eta(summary_df, figures_dir):
    if summary_df.empty:
        return None
    plt.figure(figsize=(8, 5))
    x = summary_df["eta_over_2pi_GHz"]
    plt.plot(x, summary_df["success_rate"], marker="o", label="success rate")
    plt.plot(x, summary_df["trap_ratio"], marker="s", label="trap ratio")
    plt.xlabel("eta / (2 pi GHz)")
    plt.ylabel("Rate")
    plt.ylim(-0.05, 1.05)
    plt.title("Success and trap ratio vs crosstalk")
    plt.grid(True, alpha=0.3)
    plt.legend()
    path = Path(figures_dir) / "success_trap_vs_eta.png"
    _save_current_figure(path)
    return path


def plot_restart_fidelity_scatter(df, figures_dir):
    if df.empty:
        return None
    plt.figure(figsize=(8, 5))
    for used_seed, group in df.groupby("used_physical_seed"):
        label = "physical seeds" if used_seed else "random seeds"
        marker = "o" if used_seed else "x"
        plt.scatter(
            group["eta_over_2pi_GHz"],
            group["final_fidelity"],
            alpha=0.65,
            label=label,
            marker=marker,
        )
    plt.axhline(SUCCESS_THRESHOLD_ANALYSIS, color="red", linestyle="--", label=f"threshold {SUCCESS_THRESHOLD_ANALYSIS}")
    plt.xlabel("eta / (2 pi GHz)")
    plt.ylabel("Final fidelity")
    plt.title("Restart-level final fidelity")
    plt.grid(True, alpha=0.3)
    plt.legend()
    path = Path(figures_dir) / "restart_fidelity_scatter.png"
    _save_current_figure(path)
    return path


def plot_optimizer_effort_vs_eta(summary_df, figures_dir):
    if summary_df.empty:
        return None
    plt.figure(figsize=(8, 5))
    x = summary_df["eta_over_2pi_GHz"]
    plt.plot(x, summary_df["mean_iterations"], marker="o", label="mean iterations")
    plt.plot(x, summary_df["median_iterations"], marker="s", label="median iterations")
    plt.xlabel("eta / (2 pi GHz)")
    plt.ylabel("Iterations")
    plt.title("Optimizer effort vs crosstalk")
    plt.grid(True, alpha=0.3)
    plt.legend()
    path = Path(figures_dir) / "optimizer_effort_vs_eta.png"
    _save_current_figure(path)
    return path


def plot_stop_reason_counts(summary_df, figures_dir):
    if summary_df.empty:
        return None
    stop_columns = [column for column in summary_df.columns if column.startswith("stop_") and column.endswith("_count")]
    if not stop_columns:
        return None
    plot_df = summary_df[stop_columns].copy()
    labels = [column[len("stop_") : -len("_count")] for column in stop_columns]
    plot_df.columns = labels
    ax = plot_df.plot(kind="bar", stacked=True, figsize=(10, 5))
    ax.set_xlabel("Eta index")
    ax.set_ylabel("Count")
    ax.set_title("Stop reason counts by eta")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(title="stop reason", bbox_to_anchor=(1.05, 1), loc="upper left")
    path = Path(figures_dir) / "stop_reason_counts.png"
    _save_current_figure(path)
    return path


def _parameter_group_label(column):
    if column.endswith("_A") or column.startswith("A"):
        return "amplitudes", "Amplitude"
    if "width" in column:
        return "widths", "Width (ns)"
    if "center" in column:
        return "centers", "Center (ns)"
    return "parameters", "Parameter value"


def plot_best_parameters_vs_eta(summary_df, figures_dir):
    if summary_df.empty:
        return []
    paths = []
    x = summary_df["eta_over_2pi_GHz"]
    best_parameter_columns = [
        col for col in summary_df.columns
        if col.startswith("best_")
        and col not in {"best_fidelity", "best_restart", "best_seed", "best_seed_type", "best_used_physical_seed", "best_stop_reason", "best_count"}
        and pd.api.types.is_numeric_dtype(summary_df[col])
    ]
    grouped = defaultdict(list)
    for column in best_parameter_columns:
        group_name, ylabel = _parameter_group_label(column.replace("best_", ""))
        grouped[(group_name, ylabel)].append(column)

    for (group_name, ylabel), columns in grouped.items():
        if not columns:
            continue
        plt.figure(figsize=(10, 5))
        for column in columns:
            label = column.replace("best_", "")
            plt.plot(x, summary_df[column], marker="o", label=label)
        plt.xlabel("eta / (2 pi GHz)")
        plt.ylabel(ylabel)
        plt.title(f"Best-restart piecewise {group_name} vs crosstalk")
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=8, ncol=2)
        path = Path(figures_dir) / f"best_parameters_vs_eta_{group_name}.png"
        _save_current_figure(path)
        paths.append(path)
    return paths


def plot_convergence_traces_for_eta(records, eta_index, figures_dir, max_traces=100, output_name=None):

    selected = [record for record in records if safe_int(record.get("eta_index"), default=-1) == int(eta_index)]

    if not selected:

        return None

    selected = selected[: int(max_traces)]

    figure_width = 8
    figure_height = 5
    label_size = 14
    tick_size = 12

    fig = plt.figure(figsize=(figure_width, figure_height))

    for record in selected:

        trace = [safe_float(value) for value in (record.get("record_cost") or [])]

        trace = [value for value in trace if math.isfinite(value)]

        if trace:

            plt.plot(trace)

    plt.xlabel("Optimization Iterations", fontsize=label_size)

    plt.ylabel("Fidelity", fontsize=label_size)

    eta_val = safe_float(selected[0].get("eta"), default=np.nan)
    GHz_value = globals().get("GHz", 1.0)
    eta_ghz = eta_val / (2 * np.pi * GHz_value) if math.isfinite(eta_val) else eta_index

    plt.title(
        rf"Loss = SUM; Crosstalk = {eta_ghz:.3g} GHz; "
        rf"Opt$_{{thresh}}$ = 1e-8; Iter$_{{Max}}$ = 1000"
    )

    plt.xticks(fontsize=tick_size)

    plt.yticks(fontsize=tick_size)

    if output_name is None:

        output_name = f"convergence_traces_eta_{int(eta_index):03d}.jpg"

    path = Path(figures_dir) / output_name

    path.parent.mkdir(parents=True, exist_ok=True)

    fig.savefig(path, bbox_inches="tight", dpi=300)

    plt.show()

    return path

# Save Outputs and Report

Save cleaned tables, validation reports, figures, and a short markdown report into `GaussianGRAPEAnalysis_stage2_v1/`.


In [ ]:
def save_stage2_tables(df_records, summary_df, grid_json_df, grid_csv_df, validation_report, out_dir):
    out_dir = Path(out_dir)
    tables_dir = out_dir / "tables"
    tables_dir.mkdir(parents=True, exist_ok=True)

    df_records.to_csv(tables_dir / "restart_records.csv", index=False)
    summary_df.to_csv(tables_dir / "eta_summary_from_records.csv", index=False)
    summary_df.to_json(tables_dir / "eta_summary_from_records.json", orient="records", indent=2)

    if not grid_json_df.empty:
        grid_json_df.to_csv(tables_dir / "grid_summary_loaded.csv", index=False)
        validation_report["grid_summary_source"] = "grid_summary.json"
    elif not grid_csv_df.empty:
        grid_csv_df.to_csv(tables_dir / "grid_summary_loaded.csv", index=False)
        validation_report["grid_summary_source"] = "grid_summary.csv"
    else:
        # Keep the requested grid_summary_loaded.csv output available, but mark
        # clearly that it was reconstructed from raw restart records rather than
        # loaded from a root-level grid summary file.
        summary_df.to_csv(tables_dir / "grid_summary_loaded.csv", index=False)
        summary_df.to_csv(tables_dir / "grid_summary_reconstructed_from_records.csv", index=False)
        summary_df.to_json(
            tables_dir / "grid_summary_reconstructed_from_records.json",
            orient="records",
            indent=2,
        )
        validation_report["grid_summary_source"] = "reconstructed_from_restart_records"
        validation_report.setdefault("tables", {})["grid_summary_reconstructed_from_records"] = True
        validation_report.setdefault("notes", []).append(
            "Root grid_summary.json/csv were not found; saved reconstructed grid "
            "summary from eta restart records."
        )

    with (tables_dir / "validation_report.json").open("w", encoding="utf-8") as handle:
        json.dump(make_json_safe(validation_report), handle, indent=2, allow_nan=False)

    try:
        df_records.to_parquet(tables_dir / "restart_records.parquet", index=False)
        validation_report.setdefault("tables", {})["restart_records_parquet"] = True
    except Exception as exc:
        validation_report.setdefault("tables", {})["restart_records_parquet"] = False
        validation_report.setdefault("warnings", []).append(f"Skipped parquet export: {exc}")
        with (tables_dir / "validation_report.json").open("w", encoding="utf-8") as handle:
            json.dump(make_json_safe(validation_report), handle, indent=2, allow_nan=False)


def _format_maybe(value, digits=6):
    value = safe_float(value)
    return f"{value:.{digits}f}" if math.isfinite(value) else "n/a"


def generate_stage2_report(results_root, out_dir, validation_report, df_records, summary_df):
    out_dir = Path(out_dir)
    report_path = out_dir / "stage2_report.md"
    out_dir.mkdir(parents=True, exist_ok=True)

    eta0 = summary_df[summary_df["eta_index"] == summary_df["eta_index"].min()] if not summary_df.empty else pd.DataFrame()
    etamax = summary_df[summary_df["eta_index"] == summary_df["eta_index"].max()] if not summary_df.empty else pd.DataFrame()
    eta0_row = eta0.iloc[0] if len(eta0) else {}
    etamax_row = etamax.iloc[0] if len(etamax) else {}
    success_decreases = False
    if len(summary_df) >= 2:
        success_decreases = bool(summary_df["success_rate"].iloc[-1] < summary_df["success_rate"].iloc[0])

    stop_counts = Counter()
    if not df_records.empty and "stop_reason_sanitized" in df_records.columns:
        stop_counts.update(df_records["stop_reason_sanitized"].dropna().tolist())
    common_stop_reasons = ", ".join(f"{reason}: {count}" for reason, count in stop_counts.most_common(5))

    expected_restarts = summary_df["n_restarts"].tolist() if "n_restarts" in summary_df.columns else []
    lines = [
        "# Piecewise GRAPE Stage 2 Analysis Report",
        "",
        f"- Results root: `{results_root}`",
        f"- Analysis output: `{out_dir}`",
        f"- Eta folders found: {validation_report.get('n_eta_found', 0)} / {validation_report.get('n_eta_expected', 'n/a')}",
        f"- Restart records loaded: {len(df_records)}",
        f"- Restarts per eta: {expected_restarts}",
        f"- grid_summary.json found: {validation_report.get('root_files', {}).get('grid_summary.json', False)}",
        f"- grid summary source: {validation_report.get('grid_summary_source', 'unknown')}",
        "",
        "## Key Findings",
        "",
        f"- eta=0 best fidelity: {_format_maybe(eta0_row.get('best_fidelity') if hasattr(eta0_row, 'get') else np.nan)}",
        f"- eta=0 success rate: {_format_maybe(eta0_row.get('success_rate') if hasattr(eta0_row, 'get') else np.nan, 3)}",
        f"- highest-eta best fidelity: {_format_maybe(etamax_row.get('best_fidelity') if hasattr(etamax_row, 'get') else np.nan)}",
        f"- highest-eta success rate: {_format_maybe(etamax_row.get('success_rate') if hasattr(etamax_row, 'get') else np.nan, 3)}",
        f"- success rate decreases overall: {success_decreases}",
        f"- most common stop reasons: {common_stop_reasons or 'n/a'}",
        "",
        "## Notes",
        "",
        "- Stage 2 only analyzes first-level sweep statistics.",
        "- It does not identify distinct optima.",
        "- Endpoint clustering is Stage 3.",
        "- 3D landscape slices are later and should use representative optima selected from Stage 3.",
        "- `best_fidelity` tells whether at least one good piecewise pulse was found for each eta.",
        "- `mean_fidelity` and `median_fidelity` measure typical restart quality.",
        "- `success_rate` estimates how accessible the high-fidelity basin is under the chosen finite-difference piecewise-GRAPE optimizer.",
        "- `trap_ratio = 1 - success_rate` estimates how often valid restarts end below threshold.",
        "- Physical-seed and random-seed success rates should be reported separately.",
        "- Stop reasons help distinguish landscape difficulty from optimizer artifacts.",
    ]
    report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return report_path


def _generate_figures(df_records, summary_df, records, figures_dir, n_eta_expected, validation_report):
    figures_dir = Path(figures_dir)
    figures_dir.mkdir(parents=True, exist_ok=True)
    generated = []
    skipped = []

    for plotter in [
        plot_fidelity_vs_eta,
        plot_success_trap_vs_eta,
        plot_optimizer_effort_vs_eta,
        plot_stop_reason_counts,
    ]:
        try:
            path = plotter(summary_df, figures_dir)
            generated.append(str(path)) if path else skipped.append(plotter.__name__)
        except Exception as exc:
            skipped.append(f"{plotter.__name__}: {exc}")

    try:
        path = plot_restart_fidelity_scatter(df_records, figures_dir)
        generated.append(str(path)) if path else skipped.append("plot_restart_fidelity_scatter")
    except Exception as exc:
        skipped.append(f"plot_restart_fidelity_scatter: {exc}")

    try:
        paths = plot_best_parameters_vs_eta(summary_df, figures_dir)
        generated.extend(str(path) for path in paths)
        if not paths:
            skipped.append("plot_best_parameters_vs_eta")
    except Exception as exc:
        skipped.append(f"plot_best_parameters_vs_eta: {exc}")

    eta_indices = sorted(df_records["eta_index"].dropna().astype(int).unique()) if not df_records.empty else []
    if eta_indices:
        trace_specs = [
            (eta_indices[0], f"convergence_traces_eta_{eta_indices[0]:03d}.png"),
            (eta_indices[len(eta_indices) // 2], "convergence_traces_mid_eta.png"),
            (eta_indices[-1], f"convergence_traces_eta_{eta_indices[-1]:03d}.png"),
        ]
        trace_specs = list(dict.fromkeys(trace_specs))
    else:
        trace_specs = []
    for eta_index, filename in trace_specs:
        try:
            path = plot_convergence_traces_for_eta(records, eta_index, figures_dir, output_name=filename)
            generated.append(str(path)) if path else skipped.append(filename)
        except Exception as exc:
            skipped.append(f"{filename}: {exc}")

    validation_report["figures_generated"] = generated
    validation_report["figures_skipped"] = skipped

In [ ]:
def run_stage2_analysis(
    results_root=RESULTS_ROOT,
    analysis_out=ANALYSIS_OUT,
    n_eta_expected=N_ETA_EXPECTED,
    success_threshold=SUCCESS_THRESHOLD_ANALYSIS,
):
    results_root = Path(results_root)
    analysis_out = Path(analysis_out)
    tables_dir = analysis_out / "tables"
    figures_dir = analysis_out / "figures"
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    validation_report = validate_results_structure(results_root, n_eta_expected=n_eta_expected)
    grid_summary_json_df, grid_summary_csv_df = load_grid_summary(results_root)
    records = load_restart_records(results_root, n_eta_expected=n_eta_expected)
    df_records = restart_records_to_dataframe(records, success_threshold=success_threshold)
    summary_df = summarize_by_eta_from_records(df_records, success_threshold=success_threshold)
    saved_summary_df = load_summary_rows(results_root, n_eta_expected=n_eta_expected)
    comparison_report = compare_computed_vs_saved_summary(summary_df, saved_summary_df)

    validation_report["n_restart_records_loaded"] = int(len(records))
    validation_report["n_restart_rows"] = int(len(df_records))
    validation_report["n_summary_rows_computed"] = int(len(summary_df))
    validation_report["grid_summary_json_loaded"] = not grid_summary_json_df.empty
    validation_report["grid_summary_csv_loaded"] = not grid_summary_csv_df.empty
    if not grid_summary_json_df.empty:
        validation_report["grid_summary_source"] = "grid_summary.json"
    elif not grid_summary_csv_df.empty:
        validation_report["grid_summary_source"] = "grid_summary.csv"
    else:
        validation_report["grid_summary_source"] = "reconstructed_from_restart_records"
    validation_report["summary_comparison"] = comparison_report
    validation_report.setdefault("warnings", []).extend(comparison_report.get("warnings", []))

    _generate_figures(df_records, summary_df, records, figures_dir, n_eta_expected, validation_report)
    save_stage2_tables(
        df_records,
        summary_df,
        grid_summary_json_df,
        grid_summary_csv_df,
        validation_report,
        analysis_out,
    )
    report_path = generate_stage2_report(results_root, analysis_out, validation_report, df_records, summary_df)
    validation_report["stage2_report"] = str(report_path)

    with (tables_dir / "validation_report.json").open("w", encoding="utf-8") as handle:
        json.dump(make_json_safe(validation_report), handle, indent=2, allow_nan=False)

    return {
        "records": records,
        "df_records": df_records,
        "summary_df": summary_df,
        "grid_summary_json_df": grid_summary_json_df,
        "grid_summary_csv_df": grid_summary_csv_df,
        "validation_report": validation_report,
    }

# Run Analysis

`best_fidelity` tells whether at least one good Gaussian pulse was found for each eta. `mean_fidelity` and `median_fidelity` measure typical restart quality. `success_rate` estimates high-fidelity basin accessibility, while `trap_ratio = 1 - success_rate` estimates below-threshold endpoints. Physical-seed and random-seed rates should be interpreted separately. 


In [ ]:
# Run Stage 2 analysis

stage2 = run_stage2_analysis(
    results_root=RESULTS_ROOT,
    analysis_out=ANALYSIS_OUT,
    n_eta_expected=N_ETA_EXPECTED,
    success_threshold=SUCCESS_THRESHOLD_ANALYSIS,
)

df_records = stage2["df_records"]
summary_df = stage2["summary_df"]

display(df_records.head())
display(summary_df)
